### Imports

In [ ]:
import numpy as np
import os

from cgrbptools import CGRBPTopology
from cgrbptools import ConfBuilder

---
---
# Homogenous Model
---
---

### Set output name

In [ ]:
base_filename = "Test/homog_example"

outdir = os.path.dirname(base_filename)
if outdir:
    os.makedirs(outdir, exist_ok=True)

## Set model parameters

In [ ]:
num_monomers = 400

monomer_id = 'A'

# set intrinsic shape of the junctions: rotations and translations (in rad and nm)
intrinsic_twist = np.deg2rad(0.)
intrinsic_tilt = 0.
intrinsic_roll = 0.
disc_len = 3.4

# bending and twist stiffness (in nm)
A = 40  # bending_stiffness
C = 100 # twist_stiffness
D = 200*disc_len # stretch_stiffness

# init groundstate and stiffness matrix
groundstate = np.array([intrinsic_tilt, intrinsic_roll, intrinsic_twist, 0.0, 0.0, disc_len])
local_stiffness = np.array([A / disc_len, A / disc_len, C / disc_len, D / disc_len, D / disc_len, D / disc_len])
nonlocal_couplings = None # no non-local couplings in this example

# closed topology
closed = True

# optional FENE bond 
# Set to (K, Rc, R0) to add a native FENE bond, or None to disable.
fene_coeffs = np.array([500/disc_len**2, 1.15*disc_len, 1.30*disc_len])  # (K, Rc, R0) for the FENE bond
fene_sim_units = False # fene coefficients expressed in simulation units (True) or in nm/kT

# formatting / units
decimals    = 4      # decimal places in the written coefficients
mass        = 1.0    # mass per bead
unit_length = 3.4    # nm per length unit (1.0 -> keep nm). If equal to disc_len, the junctions are written in units of disc_len. The translational stiffness is rescaled accordingly.
unit_energy = 1.0    # kT per energy unit

### Initialize Topology

In [3]:
num_monomer_steps = num_monomers if closed else num_monomers - 1

##################################################
########## Generate topology #####################
topol = CGRBPTopology(
    decimals=decimals,
    closed=closed,
)

# Optional native FENE bond (set before parameters so it is baked into the
# bond coefficients when couplings are built).
if fene_coeffs is not None:
    topol.set_fene(*fene_coeffs, sim_units=fene_sim_units)

# Homogeneous elasticity: single groundstate vector + single local stiffness
# block tiled across all NBPS junctions, plus optional nonlocal couplings.
# The coupling range is inferred from the largest offset in the dict.
topol.homogeneous_params(
    groundstate=groundstate,
    local_stiffness=local_stiffness,
    nbps=num_monomer_steps,
    nonlocal_couplings=nonlocal_couplings,
    closed=closed,
)

topol.set_unit_energy(unit_energy)
topol.set_unit_length(unit_length)

##################################################
########## Assign a (dummy) sequence #############
# The homogeneous model is sequence-independent, but the writers still need
# one label per atom. Use a uniform poly-A sequence of the right length
# (one character per atom, so length == number of atoms == topol.nbp).
seq = monomer_id * topol.nbp
topol.set_sequence(seq, chars_per_atom=1)

Initializing couplings...
Initializing couplings...


### Write Database file

In [4]:
##################################################
########## Write topology database ###############
topol.write_database(base_filename, add_extension=True)

# Build Configuration

### Chose Type

In [5]:
# options: 'circular', 'straight', 'gs', 'tresfoil'  
conf_type = 'circ'
conf_type = 'straight'
conf_type = 'tresfoil'

box_extension = 0.05
square_box = True
excess_link = 20.0

### Built-in configuration type

In [6]:
if conf_type in ConfBuilder.mapping.keys():
    conf = ConfBuilder.build(
        topol,
        conf_type,
        mass=mass,
        excess_link=excess_link,
    )
    box = conf.extended_bounds(box_extension, square_box=square_box)

    conf.write_datafile(
        base_filename,
        topol,
        box=box,
        hybrid=False,
        include_coeffs=False,
        add_extension=True,
        box_decimals=decimals,
        overwrite=True,
    )
    conf.visualize_xyz(base_filename)

    if conf_type in ['circ', 'circular'] and not closed:
        print("Warning: circular configuration requested for an open topology. The ends will be connected in the configuration.")

### Parse custom configuration via spline interpolation of tracepoints

### Tresfoil config

In [7]:
def trefoil_tracepoints(length: float, spacing: float) -> np.ndarray:
    """
    Tracepoints along a trefoil knot, equally spaced along its contour.

    The knot is the curve (sin t + 2 sin 2t, cos t - 2 cos 2t, -sin 3t), t in [0, 2 pi), scaled to
    the contour length length. Since the curve is closed, the contour is divided into
    N = round(length / spacing) equal parts, i.e. the actual spacing is length / N. The first
    tracepoint is not repeated at the end.

    Parameters
    ----------
    length : float
        Total contour length of the knot.
    spacing : float
        Distance between consecutive tracepoints along the contour (same unit as length).

    Returns
    -------
    np.ndarray
        Tracepoints, shape (N, 3).
    """
    if not (np.isfinite(length) and length > 0 and np.isfinite(spacing) and spacing > 0):
        raise ValueError(f'length and spacing must be positive, got {length} and {spacing}.')
    num = int(round(length / spacing))
    if num < 3:
        raise ValueError(f'length / spacing gives {num} tracepoints, at least 3 are required.')

    # arc length s(t) on a fine grid (trapezoidal rule), inverted by interpolation
    t = np.linspace(0.0, 2 * np.pi, 2**16 + 1)
    speed = np.sqrt(
        (np.cos(t) + 4 * np.cos(2 * t)) ** 2
        + (-np.sin(t) + 4 * np.sin(2 * t)) ** 2
        + (3 * np.cos(3 * t)) ** 2
    )
    s = np.concatenate([[0.0], np.cumsum(0.5 * (speed[1:] + speed[:-1]) * np.diff(t))])
    tk = np.interp(np.arange(num) * s[-1] / num, s, t)

    points = np.column_stack([
        np.sin(tk) + 2 * np.sin(2 * tk),
        np.cos(tk) - 2 * np.cos(2 * tk),
        -np.sin(3 * tk),
    ])
    return points * (length / s[-1])


In [14]:
if conf_type == 'tresfoil':
    points = trefoil_tracepoints(num_monomers*disc_len, disc_len)
    conf = ConfBuilder.from_tracepoints(
        topol,
        points,
        excess_link=excess_link,
        mass=mass,
        rescale=True,
    )

    box = conf.extended_bounds(box_extension, square_box=square_box)
    
    conf.write_datafile(
        base_filename,
        topol,
        box=box,
        hybrid=False,
        include_coeffs=False,
        add_extension=True,
        box_decimals=decimals,
        overwrite=True,
    )
    conf.visualize_xyz(base_filename)

    if not closed:
        print("Warning: trefoil configuration requested for an open topology. The ends will be connected in the configuration.")

### Example for importing externally generated poses

In [ ]:
if conf_type == 'tresfoil':

    # generate poses with the same routine as before
    points = trefoil_tracepoints(num_monomers*disc_len, disc_len)
    conf = ConfBuilder.from_tracepoints(
        topol,
        points,
        excess_link=excess_link,
        mass=mass,
        rescale=True,
    )
    # export poses
    poses = conf.poses.copy()

    # import poses into ConfBuilder
    conf = ConfBuilder.from_poses(
        poses,
        topology=topol,
        mass=mass
    )

    box = conf.extended_bounds(box_extension, square_box=square_box)
    conf.write_datafile(
        base_filename,
        topol,
        box=box,
        hybrid=False,
        include_coeffs=False,
        add_extension=True,
        box_decimals=decimals,
        overwrite=True,
    )
    conf.visualize_xyz(base_filename)
